# 05. Acrescentar pares por regra

Mantém a lista do [`04_atribuir.ipynb`](04_atribuir.ipynb) (`p ≥ T`). Censo e
CPF que **ainda não** entraram, na faixa `[THRESHOLD_REGRAS, T)`, entram por
funil: `nome_mae_phon` (igual ou prefixo de `n ≥ 2` tokens); senão
`nome_completo_phon` + CEP; senão primeiro+último fonéticos + CEP. Cada extra
conta só na primeira regra que passar. Mesmo desempate do 04 (CEP + mãe).
Mesmo teto: até `MAX_CENSOS_POR_CPF`
Censos por CPF **entre os extra**; CPF já no 04 não é reutilizado.

Saída: `splink_atribuicao_regras.parquet` (04 ∪ extra). O parquet do 04 não muda.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO,
    SPLINK_ATRIBUICAO_REGRAS,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    THRESHOLD_AVALIACAO,
    THRESHOLD_REGRAS,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T = THRESHOLD_AVALIACAO
T_REGRAS = THRESHOLD_REGRAS

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('T:', T, '| T_REGRAS:', T_REGRAS, '| max Censos/CPF:', MAX_CENSOS_POR_CPF)


Predictions do 02b e lista do 04. O 05 não depende das células do 03.


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE atribuicao_04 AS
SELECT unique_id_censo, unique_id_cpf, match_probability,
    'threshold' AS regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')
n_04 = con.execute('SELECT COUNT(*) FROM atribuicao_04').fetchone()[0]
print('Lista 04:', f'{n_04:,}')


Melhor CPF na faixa `[T_REGRAS, T)`, só Censo/CPF fora do 04. Empate: CEP e
mãe fonética (igual ou prefixo `n ≥ 2`), depois `unique_id_cpf`.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_faixa AS
SELECT p.unique_id_censo, p.unique_id_cpf, p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T_REGRAS}
  AND p.match_probability < {T}
  AND p.unique_id_censo NOT IN (SELECT unique_id_censo FROM atribuicao_04)
  AND p.unique_id_cpf NOT IN (SELECT unique_id_cpf FROM atribuicao_04)
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        (
            CAST(ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)
            + CAST(
                ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
                AND (
                    ca.nome_mae_phon = pb.nome_mae_phon
                    OR (
                        least(
                            len(string_split(ca.nome_mae_phon, ' ')),
                            len(string_split(pb.nome_mae_phon, ' '))
                        ) >= 2
                        AND list_slice(
                            string_split(ca.nome_mae_phon, ' '),
                            1,
                            least(
                                len(string_split(ca.nome_mae_phon, ' ')),
                                len(string_split(pb.nome_mae_phon, ' '))
                            )
                        ) = list_slice(
                            string_split(pb.nome_mae_phon, ' '),
                            1,
                            least(
                                len(string_split(ca.nome_mae_phon, ' ')),
                                len(string_split(pb.nome_mae_phon, ' '))
                            )
                        )
                    )
                ) AS INTEGER
            )
        ) DESC,
        p.unique_id_cpf
) = 1
''')
n_faixa = con.execute('SELECT COUNT(*) FROM melhor_faixa').fetchone()[0]
print('Censos na faixa (fora do 04):', f'{n_faixa:,}')


Funil (exclui o anterior): `nome_mae_phon` (igual ou prefixo de `n ≥ 2` tokens);
senão nome completo fonético + CEP; senão primeiro+último fonéticos + CEP.
Nulo não conta como igual. Um token só (`MARIA` vs `MARIA SILVA`) não prefixa.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE extra_regras AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
             AND (
                 ca.nome_mae_phon = pb.nome_mae_phon
                 OR (
                     least(
                         len(string_split(ca.nome_mae_phon, ' ')),
                         len(string_split(pb.nome_mae_phon, ' '))
                     ) >= 2
                     AND list_slice(
                         string_split(ca.nome_mae_phon, ' '),
                         1,
                         least(
                             len(string_split(ca.nome_mae_phon, ' ')),
                             len(string_split(pb.nome_mae_phon, ' '))
                         )
                     ) = list_slice(
                         string_split(pb.nome_mae_phon, ' '),
                         1,
                         least(
                             len(string_split(ca.nome_mae_phon, ' ')),
                             len(string_split(pb.nome_mae_phon, ' '))
                         )
                     )
                 )
             )
            THEN 'mae_phon'
        WHEN ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
             AND ca.nome_completo_phon = pb.nome_completo_phon
             AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            THEN 'nome_cep'
        WHEN ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL
             AND ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon IS NOT NULL AND pb.ultimo_nome_phon IS NOT NULL
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon
             AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            THEN 'pontas_cep'
    END AS regra
FROM melhor_faixa m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE (
    ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    AND (
        ca.nome_mae_phon = pb.nome_mae_phon
        OR (
            least(
                len(string_split(ca.nome_mae_phon, ' ')),
                len(string_split(pb.nome_mae_phon, ' '))
            ) >= 2
            AND list_slice(
                string_split(ca.nome_mae_phon, ' '),
                1,
                least(
                    len(string_split(ca.nome_mae_phon, ' ')),
                    len(string_split(pb.nome_mae_phon, ' '))
                )
            ) = list_slice(
                string_split(pb.nome_mae_phon, ' '),
                1,
                least(
                    len(string_split(ca.nome_mae_phon, ' ')),
                    len(string_split(pb.nome_mae_phon, ' '))
                )
            )
        )
    )
)
OR (
    ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
    AND ca.nome_completo_phon = pb.nome_completo_phon
    AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
)
OR (
    ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL
    AND ca.primeiro_nome_phon = pb.primeiro_nome_phon
    AND ca.ultimo_nome_phon IS NOT NULL AND pb.ultimo_nome_phon IS NOT NULL
    AND ca.ultimo_nome_phon = pb.ultimo_nome_phon
    AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
)
''')


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE extra_unicas AS
SELECT e.*
FROM extra_regras e
JOIN (
    SELECT unique_id_cpf
    FROM extra_regras
    GROUP BY 1
    HAVING COUNT(*) <= {MAX_CENSOS_POR_CPF}
) c ON c.unique_id_cpf = e.unique_id_cpf
''')

con.execute('''
CREATE OR REPLACE TABLE associacoes_com_regras AS
SELECT unique_id_censo, unique_id_cpf, match_probability, regra
FROM atribuicao_04
UNION ALL
SELECT unique_id_censo, unique_id_cpf, match_probability, regra
FROM extra_unicas
''')


Amostra dos extra aceitos e de quem ficou na faixa sem regra.


In [ ]:
display(con.execute(f'''
SELECT
    e.regra,
    e.match_probability,
    e.unique_id_censo,
    e.unique_id_cpf,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.nome_mae_phon AS mae_phon_censo,
    pb.nome_mae_phon AS mae_phon_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf
FROM extra_unicas e
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = e.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = e.unique_id_cpf
ORDER BY e.match_probability DESC
LIMIT 20
''').df())

display(con.execute(f'''
SELECT
    m.match_probability,
    m.unique_id_censo,
    m.unique_id_cpf,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.nome_mae_phon AS mae_phon_censo,
    pb.nome_mae_phon AS mae_phon_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf
FROM melhor_faixa m
LEFT JOIN extra_regras e ON e.unique_id_censo = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE e.unique_id_censo IS NULL
ORDER BY m.match_probability DESC
LIMIT 20
''').df())


Diagnóstico (não entra na lista): quantos a mais o `nome_mae` literal pegaria;
quantos teriam `nome_completo_phon` + município com CEP nulo.


In [ ]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n_faixa_sem_regra,
    COUNT(*) FILTER (
        WHERE ca.nome_mae IS NOT NULL AND pb.nome_mae IS NOT NULL
          AND ca.nome_mae = pb.nome_mae
    ) AS n_mae_exact_extra,
    COUNT(*) FILTER (
        WHERE (ca.cep IS NULL OR pb.cep IS NULL)
          AND ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
          AND ca.nome_completo_phon = pb.nome_completo_phon
          AND ca.cod_municipio IS NOT NULL AND pb.cod_municipio IS NOT NULL
          AND ca.cod_municipio = pb.cod_municipio
    ) AS n_nome_municipio_cep_nulo
FROM melhor_faixa m
LEFT JOIN extra_regras e ON e.unique_id_censo = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE e.unique_id_censo IS NULL
''').df())


Funil: lista do 04, depois cada regra extra excluindo quem já entrou na anterior.


In [ ]:
funil = con.execute(f'''
SELECT
    ordem,
    etapa,
    n,
    SUM(n) OVER (ORDER BY ordem) AS acumulado
FROM (
    SELECT
        1 AS ordem,
        'p ≥ {T} (melhor par, n_censo ≤ {MAX_CENSOS_POR_CPF})' AS etapa,
        (SELECT COUNT(*) FROM atribuicao_04) AS n
    UNION ALL
    SELECT
        2,
        '+ mae_phon',
        (SELECT COUNT(*) FROM extra_unicas WHERE regra = 'mae_phon')
    UNION ALL
    SELECT
        3,
        '+ nome_cep (sem mae_phon)',
        (SELECT COUNT(*) FROM extra_unicas WHERE regra = 'nome_cep')
    UNION ALL
    SELECT
        4,
        '+ pontas_cep (sem mae_phon, sem nome_cep)',
        (SELECT COUNT(*) FROM extra_unicas WHERE regra = 'pontas_cep')
)
ORDER BY ordem
''').df()
display(funil)
print('Lista final:', f'{int(funil["acumulado"].iloc[-1]):,}')


In [ ]:
print(
    'Exportado:',
    export_parquet(con, 'associacoes_com_regras', path=SPLINK_ATRIBUICAO_REGRAS),
)
con.close()
